# Run Results Diagnostics

This notebook analyzes a single simulation run using the artifacts that are actually present in this log directory.

It is designed to be robust when the exchange log only contains final summary events. In that case, the notebook falls back to end-of-run diagnostics from `summary_log.bz2`, per-agent summary logs, and the fundamental oracle log.

What this notebook does well:
- summarizes exchange-level activity and rejection reasons
- builds a per-agent results table from `summary_log.bz2`
- highlights bankrupt / underwater agents and suspicious states
- shows fundamental oracle behavior for the run

What it cannot recover from the current logs:
- final positions per agent unless they were logged explicitly
- fill-by-fill or mark-price time series if the exchange did not log them

This root notebook auto-detects the run directory and currently falls back to `log/hip3_perp_13` when opened from the repository root.


In [ ]:
from pathlib import Path
import ast
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def is_run_dir(path: Path):
    return (path / 'summary_log.bz2').exists() and (path / 'PERP_EXCHANGE.bz2').exists()

CANDIDATE_RUN_DIRS = [
    Path('.'),
    Path('log/hip3_perp_13'),
]
RUN_DIR = next((path for path in CANDIDATE_RUN_DIRS if is_run_dir(path)), Path('.'))
SUMMARY_PATH = RUN_DIR / 'summary_log.bz2'
EXCHANGE_PATH = RUN_DIR / 'PERP_EXCHANGE.bz2'
DEPLOYER_PATH = RUN_DIR / 'ORACLE_DEPLOYER.bz2'
FUNDAMENTAL_PATHS = sorted(RUN_DIR.glob('fundamental_*.bz2'))
SPECIAL_STEMS = {'summary_log', 'PERP_EXCHANGE', 'ORACLE_DEPLOYER'}

def load_bz2(path: Path):
    if not path.exists():
        return None
    return pd.read_pickle(path, compression='bz2')

def parse_event(value):
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.startswith('{') or stripped.startswith('['):
            try:
                return ast.literal_eval(stripped)
            except Exception:
                return value
    return value

def nested_value(payload, *keys, default=0):
    cur = payload
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key, default)
    return cur

summary = load_bz2(SUMMARY_PATH)
exchange_log = load_bz2(EXCHANGE_PATH)
deployer_log = load_bz2(DEPLOYER_PATH)
fundamentals = {path.stem.replace('fundamental_', ''): load_bz2(path) for path in FUNDAMENTAL_PATHS}

print(f'Run directory: {RUN_DIR.resolve()}')
print(f'Summary log rows: {0 if summary is None else len(summary)}')
print(f'Exchange log rows: {0 if exchange_log is None else len(exchange_log)}')
print(f'Deployer log rows: {0 if deployer_log is None else len(deployer_log)}')
print(f'Fundamental logs: {list(fundamentals)}')


In [ ]:
assert summary is not None, 'summary_log.bz2 not found'

summary = summary.copy()
summary['ParsedEvent'] = summary['Event'].apply(parse_event)

exchange_activity = summary.loc[summary['EventType'] == 'PERP_EXCHANGE_ACTIVITY_SUMMARY', 'ParsedEvent']
exchange_rejections = summary.loc[summary['EventType'] == 'PERP_EXCHANGE_REJECTION_REASONS', 'ParsedEvent']
exchange_open_orders = summary.loc[summary['EventType'] == 'PERP_EXCHANGE_OPEN_ORDERS_AT_STOP', 'ParsedEvent']

exchange_activity = exchange_activity.iloc[0] if not exchange_activity.empty else {}
exchange_rejections = exchange_rejections.iloc[0] if not exchange_rejections.empty else {}
exchange_open_orders = exchange_open_orders.iloc[0] if not exchange_open_orders.empty else {}

print('Exchange log event types present:')
if exchange_log is not None and len(exchange_log) > 0:
    print(exchange_log['EventType'].value_counts().to_string())
else:
    print('  <none>')

rich_exchange_events = ['SET_ORACLE', 'MARK_PRICE_UPDATE', 'FUNDING_SETTLED', 'QUERY_MARK_PRICE']
available_exchange_events = set(exchange_log['EventType']) if exchange_log is not None else set()
missing_exchange_events = [name for name in rich_exchange_events if name not in available_exchange_events]
if missing_exchange_events:
    print()
    print('Detailed exchange time-series events are not available for this run:')
    print('  ' + ', '.join(missing_exchange_events))
    print('Notebook will use end-of-run summaries instead of pretending those series exist.')

exchange_activity_df = pd.DataFrame.from_dict(exchange_activity.get('global', {}), orient='index', columns=['count']).rename_axis('metric')
exchange_rejection_df = pd.DataFrame.from_dict(exchange_rejections.get('global', {}), orient='index', columns=['count']).rename_axis('rejection_reason')
if exchange_open_orders:
    exchange_open_df = pd.DataFrame(
        [
            {
                'symbol': symbol,
                **details,
            }
            for symbol, details in exchange_open_orders.items()
        ]
    ).set_index('symbol')
else:
    exchange_open_df = pd.DataFrame(columns=['resting', 'trigger', 'dormant_trigger'])

display(exchange_activity_df)
display(exchange_rejection_df)
display(exchange_open_df)


In [ ]:
agent_rows = []
for agent_id, group in summary.groupby('AgentID'):
    strategy = group['AgentStrategy'].dropna().iloc[0]
    if strategy == 'PerpExchangeAgent':
        continue
    row = {'AgentID': int(agent_id), 'AgentStrategy': strategy}
    for event_type in [
        'STARTING_CASH', 'FINAL_BALANCE', 'ENDING_EQUITY', 'FINAL_VALUATION',
        'PERP_ACTIVITY_SUMMARY', 'PERP_REJECTION_REASONS', 'PERP_OPEN_ORDERS_AT_STOP'
    ]:
        values = group.loc[group['EventType'] == event_type, 'ParsedEvent']
        if not values.empty:
            row[event_type] = values.iloc[0]
    agent_rows.append(row)

agents = pd.DataFrame(agent_rows).sort_values('AgentID').reset_index(drop=True)

agent_log_stems = []
for path in RUN_DIR.glob('*.bz2'):
    stem = path.stem
    if stem in SPECIAL_STEMS or stem.startswith('fundamental_'):
        continue
    agent_log_stems.append(stem)

prefix_order = {'Noise': 0, 'Momentum': 1, 'Value': 2, 'Fundamentalist': 3, 'Chartist': 4}

def stem_sort_key(stem):
    match = re.match(r'([A-Za-z]+)_(\d+)$', stem)
    if match:
        prefix, idx = match.groups()
        return (prefix_order.get(prefix, 999), prefix, int(idx))
    return (998, stem, -1)

ordered_stems = sorted(agent_log_stems, key=stem_sort_key)
if len(ordered_stems) == len(agents):
    agents['AgentName'] = pd.Series(ordered_stems, index=agents.index)
else:
    agents['AgentName'] = agents['AgentStrategy'] + '_' + agents.groupby('AgentStrategy').cumcount().astype(str)
    print('Warning: agent log count did not match summary agent count, so names were synthesized.')

for col in ['STARTING_CASH', 'FINAL_BALANCE', 'ENDING_EQUITY', 'FINAL_VALUATION']:
    if col in agents.columns:
        agents[col] = pd.to_numeric(agents[col], errors='coerce')

agents['pnl'] = agents['ENDING_EQUITY'] - agents['STARTING_CASH']
agents['pnl_pct'] = agents['pnl'] / agents['STARTING_CASH']
agents['cash_drift'] = agents['FINAL_BALANCE'] - agents['STARTING_CASH']
agents['is_negative_equity'] = agents['ENDING_EQUITY'] < 0
agents['is_negative_balance'] = agents['FINAL_BALANCE'] < 0

def symbol_payload(payload):
    if isinstance(payload, dict) and payload:
        symbol = next(iter(payload))
        return symbol, payload[symbol]
    return None, {}

activity_payloads = []
for payload in agents.get('PERP_ACTIVITY_SUMMARY', []):
    symbol, info = symbol_payload(payload)
    activity_payloads.append({
        'symbol': symbol,
        'submitted': nested_value(info, 'submitted'),
        'accepted': nested_value(info, 'accepted'),
        'rejected': nested_value(info, 'rejected'),
        'executed': nested_value(info, 'executed'),
        'cancelled': nested_value(info, 'cancelled'),
        'open_orders_at_stop': nested_value(info, 'open_orders_at_stop'),
        'status': nested_value(info, 'status', default=''),
        'rej_OI_CAP': nested_value(info, 'rejection_reasons', 'OI_CAP'),
        'rej_UNFUNDED_ACCOUNT': nested_value(info, 'rejection_reasons', 'UNFUNDED_ACCOUNT'),
        'rej_REDUCE_ONLY_NO_POSITION': nested_value(info, 'rejection_reasons', 'REDUCE_ONLY_NO_POSITION'),
        'rej_MAX_ORDER_VALUE': nested_value(info, 'rejection_reasons', 'MAX_ORDER_VALUE'),
        'rej_MIN_ORDER_VALUE': nested_value(info, 'rejection_reasons', 'MIN_ORDER_VALUE'),
    })

activity_df = pd.DataFrame(activity_payloads)
agents = pd.concat([agents, activity_df], axis=1)
agents['accept_rate'] = np.where(agents['submitted'] > 0, agents['accepted'] / agents['submitted'], np.nan)
agents['exec_per_accept'] = np.where(agents['accepted'] > 0, agents['executed'] / agents['accepted'], np.nan)
agents['bankrupt_flat'] = (agents['ENDING_EQUITY'] < 0) & (agents['status'] == 'filled')
agents['status_open_mismatch'] = (agents['open_orders_at_stop'] > 0) & (agents['status'] == 'filled')

key_cols = [
    'AgentID', 'AgentName', 'AgentStrategy', 'STARTING_CASH', 'FINAL_BALANCE', 'ENDING_EQUITY',
    'pnl', 'pnl_pct', 'submitted', 'accepted', 'rejected', 'executed', 'cancelled',
    'open_orders_at_stop', 'status', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP',
    'rej_REDUCE_ONLY_NO_POSITION', 'rej_MAX_ORDER_VALUE', 'rej_MIN_ORDER_VALUE',
    'accept_rate', 'exec_per_accept', 'bankrupt_flat', 'status_open_mismatch'
]

display(agents[key_cols].sort_values(['AgentStrategy', 'AgentID']).reset_index(drop=True))


In [ ]:
strategy_summary = agents.groupby('AgentStrategy').agg(
    agents=('AgentID', 'count'),
    negative_equity=('is_negative_equity', 'sum'),
    negative_balance=('is_negative_balance', 'sum'),
    bankrupt_flat=('bankrupt_flat', 'sum'),
    status_open_mismatch=('status_open_mismatch', 'sum'),
    mean_equity=('ENDING_EQUITY', 'mean'),
    median_equity=('ENDING_EQUITY', 'median'),
    min_equity=('ENDING_EQUITY', 'min'),
    max_equity=('ENDING_EQUITY', 'max'),
    mean_pnl=('pnl', 'mean'),
    median_pnl=('pnl', 'median'),
    mean_submitted=('submitted', 'mean'),
    mean_accepted=('accepted', 'mean'),
    mean_rejected=('rejected', 'mean'),
    mean_executed=('executed', 'mean'),
    mean_unfunded_rejections=('rej_UNFUNDED_ACCOUNT', 'mean'),
    mean_oi_cap_rejections=('rej_OI_CAP', 'mean'),
).sort_index()

display(strategy_summary)

print('Worst ending equity:')
display(agents.sort_values('ENDING_EQUITY').head(20)[['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'status', 'open_orders_at_stop', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP']])

print('Best ending equity:')
display(agents.sort_values('ENDING_EQUITY', ascending=False).head(20)[['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'status', 'open_orders_at_stop', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP']])

print('Agents with negative equity:')
display(agents.loc[agents['is_negative_equity'], ['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'status', 'open_orders_at_stop', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'rej_REDUCE_ONLY_NO_POSITION']].sort_values('ENDING_EQUITY'))

print('Suspicious summary states: status says filled but open orders remain at stop')
display(agents.loc[agents['status_open_mismatch'], ['AgentID', 'AgentName', 'AgentStrategy', 'status', 'open_orders_at_stop', 'executed', 'ENDING_EQUITY']].sort_values(['AgentStrategy', 'AgentID']))


In [ ]:
agents_plot = agents.copy()
strategy_labels = {
    'PerpNoiseAgent': 'Noise',
    'ChiarellaAgent': 'Chiarella',
}
agents_plot['StrategyLabel'] = agents_plot['AgentStrategy'].map(strategy_labels).fillna(agents_plot['AgentStrategy'])
agents_plot['rejection_rate'] = np.where(agents_plot['submitted'] > 0, agents_plot['rejected'] / agents_plot['submitted'], np.nan)
agents_plot['unfunded_share_of_rejections'] = np.where(agents_plot['rejected'] > 0, agents_plot['rej_UNFUNDED_ACCOUNT'] / agents_plot['rejected'], 0.0)
agents_plot['equity_multiple'] = np.where(agents_plot['STARTING_CASH'] > 0, agents_plot['ENDING_EQUITY'] / agents_plot['STARTING_CASH'], np.nan)

plot_groups = [(label, group.copy()) for label, group in agents_plot.groupby('StrategyLabel')]
colors = {'Noise': '#d55e00', 'Chiarella': '#0072b2'}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

box_data = [group['ENDING_EQUITY'].dropna().values for label, group in plot_groups]
box_labels = [label for label, _ in plot_groups]
box = axes[0, 0].boxplot(box_data, labels=box_labels, patch_artist=True)
for patch, label in zip(box['boxes'], box_labels):
    patch.set_facecolor(colors.get(label, '#999999'))
    patch.set_alpha(0.5)
axes[0, 0].axhline(0, color='black', linewidth=0.8)
axes[0, 0].set_title('Ending Equity By Strategy')
axes[0, 0].set_ylabel('Ending equity')
axes[0, 0].grid(True, alpha=0.3)

for label, group in plot_groups:
    axes[0, 1].scatter(group['rej_UNFUNDED_ACCOUNT'], group['ENDING_EQUITY'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[0, 1].axhline(0, color='black', linewidth=0.8)
axes[0, 1].set_title('UNFUNDED_ACCOUNT Rejections vs Ending Equity')
axes[0, 1].set_xlabel('UNFUNDED_ACCOUNT rejections')
axes[0, 1].set_ylabel('Ending equity')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

for label, group in plot_groups:
    axes[1, 0].scatter(group['accept_rate'], group['pnl'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('Acceptance Rate vs PnL')
axes[1, 0].set_xlabel('accept rate')
axes[1, 0].set_ylabel('PnL')
axes[1, 0].grid(True, alpha=0.3)

for label, group in plot_groups:
    axes[1, 1].scatter(group['rejection_rate'], group['unfunded_share_of_rejections'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[1, 1].set_title('Rejection Rate vs Share Of Rejections That Are Unfunded')
axes[1, 1].set_xlabel('rejection rate')
axes[1, 1].set_ylabel('UNFUNDED share of rejections')
axes[1, 1].set_ylim(-0.05, 1.05)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Agents with the most UNFUNDED_ACCOUNT rejections:')
display(
    agents_plot.sort_values(['rej_UNFUNDED_ACCOUNT', 'ENDING_EQUITY'], ascending=[False, True]).head(15)[
        ['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'submitted', 'accepted', 'rejected', 'accept_rate', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'status']
    ]
)

print('Agents with the highest rejection rates:')
display(
    agents_plot.sort_values(['rejection_rate', 'rej_UNFUNDED_ACCOUNT'], ascending=[False, False]).head(15)[
        ['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'submitted', 'accepted', 'rejected', 'rejection_rate', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'status']
    ]
)


In [ ]:
# Extra distribution and outlier views
plot_df = agents_plot.copy()
plot_groups = [(label, group.copy()) for label, group in plot_df.groupby('StrategyLabel')]
colors = {'Noise': '#d55e00', 'Chiarella': '#0072b2'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for label, group in plot_groups:
    axes[0, 0].hist(group['ENDING_EQUITY'].dropna(), bins=20, alpha=0.45, label=label, color=colors.get(label, '#999999'))
axes[0, 0].axvline(0, color='black', linewidth=0.8)
axes[0, 0].set_title('Ending Equity Distribution')
axes[0, 0].set_xlabel('Ending equity')
axes[0, 0].set_ylabel('Agent count')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

neg_counts = plot_df.groupby('StrategyLabel')['is_negative_equity'].sum().sort_values(ascending=False)
axes[0, 1].bar(neg_counts.index, neg_counts.values, color=[colors.get(label, '#999999') for label in neg_counts.index])
axes[0, 1].set_title('Negative-Equity Agents By Strategy')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(True, axis='y', alpha=0.3)

worst = plot_df.nsmallest(12, 'ENDING_EQUITY').sort_values('ENDING_EQUITY')
axes[1, 0].barh(worst['AgentName'], worst['ENDING_EQUITY'], color=[colors.get(label, '#999999') for label in worst['StrategyLabel']])
axes[1, 0].axvline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('Worst 12 Agents By Ending Equity')
axes[1, 0].set_xlabel('Ending equity')
axes[1, 0].grid(True, axis='x', alpha=0.3)

best = plot_df.nlargest(12, 'ENDING_EQUITY').sort_values('ENDING_EQUITY')
axes[1, 1].barh(best['AgentName'], best['ENDING_EQUITY'], color=[colors.get(label, '#999999') for label in best['StrategyLabel']])
axes[1, 1].axvline(0, color='black', linewidth=0.8)
axes[1, 1].set_title('Best 12 Agents By Ending Equity')
axes[1, 1].set_xlabel('Ending equity')
axes[1, 1].grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print('Worst-agent table used for the chart:')
display(worst[['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'submitted', 'accepted', 'rejected', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'status']])

print('Best-agent table used for the chart:')
display(best[['AgentID', 'AgentName', 'AgentStrategy', 'ENDING_EQUITY', 'FINAL_BALANCE', 'submitted', 'accepted', 'rejected', 'rej_UNFUNDED_ACCOUNT', 'rej_OI_CAP', 'status']])


In [ ]:
# Extra rejection-mix and activity plots
plot_df = agents_plot.copy()
rejection_cols = [
    'rej_UNFUNDED_ACCOUNT',
    'rej_OI_CAP',
    'rej_REDUCE_ONLY_NO_POSITION',
    'rej_MAX_ORDER_VALUE',
    'rej_MIN_ORDER_VALUE',
]
rejection_mix = plot_df.groupby('StrategyLabel')[rejection_cols].sum()
colors = {'Noise': '#d55e00', 'Chiarella': '#0072b2'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

rejection_mix.plot(kind='bar', stacked=True, ax=axes[0, 0], colormap='tab20c')
axes[0, 0].set_title('Aggregate Rejection Mix By Strategy')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(True, axis='y', alpha=0.3)
axes[0, 0].legend(loc='upper right', fontsize=8)

for label, group in plot_df.groupby('StrategyLabel'):
    axes[0, 1].scatter(group['submitted'], group['ENDING_EQUITY'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[0, 1].axhline(0, color='black', linewidth=0.8)
axes[0, 1].set_title('Submitted Orders vs Ending Equity')
axes[0, 1].set_xlabel('Submitted orders')
axes[0, 1].set_ylabel('Ending equity')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

for label, group in plot_df.groupby('StrategyLabel'):
    axes[1, 0].scatter(group['rej_OI_CAP'], group['ENDING_EQUITY'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[1, 0].axhline(0, color='black', linewidth=0.8)
axes[1, 0].set_title('OI_CAP Rejections vs Ending Equity')
axes[1, 0].set_xlabel('OI_CAP rejections')
axes[1, 0].set_ylabel('Ending equity')
axes[1, 0].grid(True, alpha=0.3)

for label, group in plot_df.groupby('StrategyLabel'):
    axes[1, 1].scatter(group['executed'], group['ENDING_EQUITY'], s=45, alpha=0.75, label=label, color=colors.get(label, '#999999'))
axes[1, 1].axhline(0, color='black', linewidth=0.8)
axes[1, 1].set_title('Executions vs Ending Equity')
axes[1, 1].set_xlabel('Executed trades')
axes[1, 1].set_ylabel('Ending equity')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Aggregate rejection mix:')
display(rejection_mix)


In [ ]:
if not fundamentals:
    print('No fundamental logs found.')
else:
    for symbol, fund in fundamentals.items():
        fund = fund.copy()
        fund.index = pd.to_datetime(fund.index)
        print(f'Fundamental summary for {symbol}:')
        display(fund['FundamentalValue'].describe().to_frame('FundamentalValue'))
        print(f'Time range: {fund.index.min()} -> {fund.index.max()}')
        print(f"First value: {fund['FundamentalValue'].iloc[0]:.4f}")
        print(f"Last value:  {fund['FundamentalValue'].iloc[-1]:.4f}")
        print(f"Max value:   {fund['FundamentalValue'].max():.4f}")
        print(f"Min value:   {fund['FundamentalValue'].min():.4f}")
        ax = fund['FundamentalValue'].plot(figsize=(14, 4), title=f'Fundamental oracle series: {symbol}', linewidth=1)
        ax.set_ylabel('Price')
        ax.grid(True, alpha=0.3)
        plt.show()

if exchange_log is not None and len(exchange_log) > 0 and 'MARK_PRICE_UPDATE' in set(exchange_log['EventType']):
    mark = exchange_log.loc[exchange_log['EventType'] == 'MARK_PRICE_UPDATE'].copy().reset_index()
    parsed = mark['Event'].str.split(',', expand=True)
    if parsed.shape[1] >= 3:
        mark['symbol'] = parsed[0]
        mark['mark_price'] = parsed[1].astype(float)
        mark['oracle_price'] = parsed[2].astype(float)
        mark['EventTime'] = pd.to_datetime(mark['EventTime'])
        mark['premium_bps'] = (mark['mark_price'] - mark['oracle_price']) / mark['oracle_price'] * 10000
        display(mark[['EventTime', 'symbol', 'mark_price', 'oracle_price', 'premium_bps']].head())
        fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
        axes[0].plot(mark['EventTime'], mark['oracle_price'], label='oracle', linewidth=1)
        axes[0].plot(mark['EventTime'], mark['mark_price'], label='mark', linewidth=1)
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        axes[0].set_ylabel('Price')
        axes[1].plot(mark['EventTime'], mark['premium_bps'], linewidth=1)
        axes[1].axhline(0, color='black', linewidth=0.8)
        axes[1].set_ylabel('Premium (bps)')
        axes[1].grid(True, alpha=0.3)
        plt.show()
else:
    print('Detailed mark/oracle exchange events are unavailable for this run, so no mark-price time series can be reconstructed from logs.')
